In [4]:
import numpy as np
import pandas as pd

file_path = 'same_side.csv'

bursts = pd.read_csv(file_path)

print(bursts.head())

       Start time        End time  High frequency (Hz)  Low frequency (Hz)  \
0  2023-1-1/01:30  2023-1-1/04:00           10000000.0             50000.0   
1  2023-1-3/06:25  2023-1-3/09:00           20000000.0             70000.0   
2  2023-1-4/03:14  2023-1-4/03:20           10000000.0            600000.0   
3  2023-1-4/05:53  2023-1-4/06:15           20000000.0            300000.0   
4  2023-1-4/09:19  2023-1-4/09:24           20000000.0            600000.0   

  GOES Flare Notes (corona loop brightening, etc.) Other notes  
0    T     T                                  Loop         NaN  
1    T     T                                  Loop         NaN  
2    F   NaN                                   NaN         NaN  
3    T     T                                  Loop         NaN  
4    F   NaN                                   NaN         NaN  


In [6]:
freq_change = bursts["High frequency (Hz)"] - bursts["Low frequency (Hz)"]
bursts["Frequency Change"] = freq_change

bursts['Start time'] = pd.to_datetime(bursts['Start time'], format='%Y-%m-%d/%H:%M', errors='coerce')
bursts['End time'] = pd.to_datetime(bursts['End time'], format='%Y-%m-%d/%H:%M', errors='coerce')
bursts['Duration (seconds)'] = (bursts['End time'] - bursts['Start time']).dt.total_seconds()

drift_rate = bursts["Frequency Change"] / bursts["Duration (seconds)"]
bursts["Drift Rate (Hz/s)"] = drift_rate
bursts.head()

,Start time,End time,High frequency (Hz),Low frequency (Hz),GOES,Flare,"Notes (corona loop brightening, etc.)",Other notes,Frequency Change,Duration (seconds),Drift Rate (Hz/s)
0,2023-01-01 01:30:00,2023-01-01 04:00:00,10000000.0,50000.0,T,T,Loop,NaN,9950000.0,9000.0,1105.555556
1,2023-01-03 06:25:00,2023-01-03 09:00:00,20000000.0,70000.0,T,T,Loop,NaN,19930000.0,9300.0,2143.010753
2,2023-01-04 03:14:00,2023-01-04 03:20:00,10000000.0,600000.0,F,NaN,NaN,NaN,9400000.0,360.0,26111.111111
3,2023-01-04 05:53:00,2023-01-04 06:15:00,20000000.0,300000.0,T,T,Loop,NaN,19700000.0,1320.0,14924.242424
4,2023-01-04 09:19:00,2023-01-04 09:24:00,20000000.0,600000.0,F,NaN,NaN,NaN,19400000.0,300.0,64666.666667


In [8]:
without_flare = bursts[(bursts["GOES"] == "T") & (bursts["Flare"] == "F")]
with_flare = bursts[(bursts["GOES"] == "T") & (bursts["Flare"] == "T")]
spot = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Spot"]
loop = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Loop"]
loop_spot = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Loop & Spot"]

In [5]:
# print length
print("All bursts length: ", bursts.shape[0])
print("Without flare length: ", without_flare.shape[0])
print("With flare length: ", with_flare.shape[0])
print("All GOES correspondence: ", without_flare.shape[0] + with_flare.shape[0])
print("All spot bursts: ", spot.shape[0])
print("All loop bursts: ", loop.shape[0])
print("All loop and spot bursts: ", loop_spot.shape[0])

All bursts length:  483
Without flare length:  54
With flare length:  216
All GOES correspondence:  270
All spot bursts:  134
All loop bursts:  64
All loop and spot bursts:  21


In [10]:
# generate pop-up plot
from pytplot import get_data
from matplotlib import pyplot as plt
%matplotlib tk
from matplotlib.colors import LogNorm

In [14]:
# duration hist line plot function
def plot_duration_hist(data, name):
    duration = data["Duration (seconds)"].dropna()
    log_bins = np.logspace(np.log10(duration.min()), np.log10(duration.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(duration, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([duration.min() * 0.8, duration.max() * 1.2])  # Add padding
    plt.title(name + ' Duration Histogram')
    plt.xlabel('Duration (s)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(duration, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [16]:
# plot line histogram all in 1 graph
def plot_combined_hist_line(datasets, labels):

    plt.figure(figsize=(10, 6))

    for data, label in zip(datasets, labels):
        duration = data["Duration (seconds)"].dropna()
        log_bins = np.logspace(np.log10(duration.min()), np.log10(duration.max()), 20)
        
        counts, bins = np.histogram(duration, bins=log_bins)
        plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', label=label, linestyle='-')

    plt.xscale('log')
    plt.xlabel('Duration (s)')
    plt.ylabel('Count')
    plt.title('Combined Duration Histogram Lines')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_combined_hist_line(
    [with_flare, without_flare, spot, loop, loop_spot], 
    ["With Flare", "Without Flare", "Spot Burst", "Loop Burst", "Loop and Spot Burst"]
)

In [9]:
# freq change hist line plot function
def plot_duration_hist(data, name):
    freq_change = data["Frequency Change"].dropna()
    log_bins = np.logspace(np.log10(freq_change.min()), np.log10(freq_change.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(freq_change, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([freq_change.min() * 0.8, freq_change.max() * 1.2])  # Add padding
    plt.title(name + ' Frequency Change Histogram')
    plt.xlabel('Freq Change (Hz)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(freq_change, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [25]:
# plot line histogram all in 1 graph
def plot_combined_hist_line(datasets, labels):

    plt.figure(figsize=(10, 6))

    for data, label in zip(datasets, labels):
        freq_change = data["Frequency Change"].dropna()
        log_bins = np.logspace(np.log10(freq_change.min()), np.log10(freq_change.max()), 20)
        
        counts, bins = np.histogram(freq_change, bins=log_bins)
        plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', label=label, linestyle='-')

    plt.xscale('log')
    plt.xlabel('Frequency Change (Hz)')
    plt.ylabel('Count')
    plt.title('Combined Freq Change Histogram Lines')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_combined_hist_line(
    [with_flare, without_flare, spot, loop, loop_spot], 
    ["With Flare", "Without Flare", "Spot Burst", "Loop Burst", "Loop and Spot Burst"]
)

In [10]:
# drift rate hist line plot function
def plot_duration_hist(data, name):
    drift_rate = data["Drift Rate (Hz/s)"].dropna()
    log_bins = np.logspace(np.log10(drift_rate.min()), np.log10(drift_rate.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(drift_rate, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([drift_rate.min() * 0.8, drift_rate.max() * 1.2])  # Add padding
    plt.title(name + ' Drift Rate Histogram')
    plt.xlabel('Drift Rate (Hz/s)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(drift_rate, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [23]:
# plot line histogram all in 1 graph
def plot_combined_hist_line(datasets, labels):

    plt.figure(figsize=(10, 6))

    for data, label in zip(datasets, labels):
        drift_rate = data["Drift Rate (Hz/s)"].dropna()
        log_bins = np.logspace(np.log10(drift_rate.min()), np.log10(drift_rate.max()), 20)
        
        counts, bins = np.histogram(drift_rate, bins=log_bins)
        plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', label=label, linestyle='-')

    plt.xscale('log')
    plt.xlabel('Drift Rate (Hz/s)')
    plt.ylabel('Count')
    plt.title('Combined Drift Rate Histogram Lines')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_combined_hist_line(
    [with_flare, without_flare, spot, loop, loop_spot], 
    ["With Flare", "Without Flare", "Spot Burst", "Loop Burst", "Loop and Spot Burst"]
)

In [11]:
# poisson noise for with flare
means = with_flare[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with flare:\n", poisson_noise)

# poisson noise for without flare
means = without_flare[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for without flare:\n", poisson_noise)

# poisson noise for with spot
means = spot[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with spot:\n", poisson_noise)

# poisson noise for with loop
means = loop[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with loop:\n", poisson_noise)

# poisson noise for with loop & spot
means = loop_spot[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with loop:\n", poisson_noise)


Poisson Noise for with flare:
 Frequency Change      3908.775739
Duration (seconds)      54.306230
Drift Rate (Hz/s)      107.207393
dtype: float64

Poisson Noise for without flare:
 Frequency Change      3781.313705
Duration (seconds)      51.164224
Drift Rate (Hz/s)      112.573515
dtype: float64

Poisson Noise for with spot:
 Frequency Change      3944.296846
Duration (seconds)      55.228156
Drift Rate (Hz/s)      100.465757
dtype: float64

Poisson Noise for with loop:
 Frequency Change      3848.132664
Duration (seconds)      51.088159
Drift Rate (Hz/s)      116.806684
dtype: float64

Poisson Noise for with loop:
 Frequency Change      3779.770716
Duration (seconds)      58.334014
Drift Rate (Hz/s)       98.129940
dtype: float64
